# 05 — Active Learning Loop

Each round:
  1. Score unlabeled pool → pick most uncertain
  2. NSFW oracle + VLM labeling on the selection
  3. Retrain (warm start)
  4. Tune thresholds + recalibrate
  5. Re-evaluate on holdout

Run 3-5 rounds. Stop when recall_NOT_ACCEPTABLE on holdout plateaus.

In [ ]:
import os, pathlib
if not pathlib.Path('zahava-local-ai-content-filter').exists():
    !git clone https://github.com/zahava-networks/zahava-local-ai-content-filter.git
os.chdir('zahava-local-ai-content-filter')
!pip install -q -r requirements.txt timm torch torchvision transformers
assert pathlib.Path('.env').exists(), 'upload .env first'

In [ ]:
# Pull latest artifacts from HF
from huggingface_hub import hf_hub_download
from pipelines.common import require_env
for f in ('collection_deduped.parquet', 'labels.parquet', 'human_review.parquet'):
    try:
        hf_hub_download(repo_id=require_env('HF_DATASET_REPO'), filename=f, repo_type='dataset', local_dir='manifests')
    except Exception as e:
        print(f'skip {f}: {e}')

In [ ]:
# Run a round. Set N manually.
from pipelines.active_learning.round_runner import run_round
metrics = run_round(round_idx=2)
print(metrics)

In [ ]:
# Plot improvement across rounds
import json, matplotlib.pyplot as plt, pathlib
h = json.loads(pathlib.Path('models/eval/history.json').read_text())
x = [r['round'] for r in h['rounds']]
rec = [r['metrics']['block_recall_not_acceptable'] for r in h['rounds']]
pre = [r['metrics']['block_precision'] for r in h['rounds']]
plt.plot(x, rec, label='recall (NOT_ACCEPTABLE)', marker='o')
plt.plot(x, pre, label='precision', marker='o')
plt.xlabel('round'); plt.ylabel('metric'); plt.legend(); plt.title('Active learning progress'); plt.show()